# LLM, prompting и RAG

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Различать предобучение, дообучение, prompting и RAG и проверять локальные источники.

**LLM** (*large language model*, большая языковая модель) строит распределение следующего токена по контексту и может продолжать текст шаг за шагом. С этим механизмом мы познакомились в [авторегрессии](../generative-models/). Теперь разберём, как меняют поведение такой модели и добавляют ей сведения из источников; большой размер сам по себе не гарантирует верность сведений.

## Четыре разных действия

Предобучение учит параметры на большом общем наборе. Дообучение меняет параметры по дополнительным примерам. Prompting меняет входную инструкцию и контекст без обновления весов. RAG, генерация с извлечением источников, сначала ищет релевантные документы и передаёт найденные фрагменты модели вместе с запросом.

Ни один из этих способов сам по себе не гарантирует правильный ответ. Инструкция может быть двусмысленной, данные устареть, поиск вернуть неподходящий фрагмент, а модель — добавить неподтверждённое. Внешний найденный текст также не становится автоматически доверенной командой для программы.

## Разобранный пример: школьный справочник

Есть две записи: «Библиотека работает до 18 часов» и «Кружок робототехники проходит по вторникам». Запрос спрашивает о библиотеке. Простейший поиск по общим словам выбирает первый фрагмент. Его нужно передать с идентификатором источника и проверить, что ответ о времени действительно опирается на него.

Если вопрос «работает ли библиотека в субботу», сведения о времени закрытия не отвечают о днях. Корректное поведение — сообщить недостаток данных, а не придумать расписание. Даже идеальный поиск по имеющимся документам не восстановит отсутствующий факт.

## Практика без внешней модели

В ноутбуке реализован только локальный поиск по маленькому авторскому справочнику. Никакая LLM не вызывается, её текст не имитируется. Для проверки полного RAG отдельно оценивают качество поиска и обоснованность каждого утверждения ответа. Сохранение источников и их версий помогает повторить разбор.

При выборе способа учитывают приватность, лицензию, стоимость запросов и задержку. Контекст имеет ограниченную длину, поэтому добавление всех документов подряд не решает задачу во всех случаях.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

В этом простом поиске совпадают только одинаковые слова после `lower` и `split`: «часы» и «часов» сами по себе не совпадут. В запросе помогает слово «библиотека». Если общих слов нет вовсе, код сообщит об отсутствии опоры вместо произвольного выбора первого документа.

In [ ]:
documents = {"library": "библиотека работает до 18 часов", "club": "кружок робототехники по вторникам"}
def tokens(text):
    return set(text.lower().split())
query = "библиотека часы"
ranked = sorted([((-len(tokens(query) & tokens(text))), key) for key, text in documents.items()])
negative_overlap, selected = ranked[0]
if negative_overlap == 0:
    raise ValueError("Нет общих слов с источниками: уточните запрос")
assert selected == "library"
assert "18" in documents[selected]
print("Найденный источник:", selected, documents[selected])
print("Это поиск; генеративная модель не запускалась")


## Типичные ошибки

Не называйте локальный поиск запуском LLM. Не подменяйте отсутствующий факт уверенным ответом.

## Задачи

### 1. Базовая

Меняет ли prompting веса?

<details>
<summary>Подсказка</summary>

Меняется вход.

</details>



### 2. Базовая

Что добавляет RAG перед ответом?

<details>
<summary>Подсказка</summary>

Сначала работа с документами.

</details>



### 3. Прикладная

Какой источник подходит вопросу о часе закрытия библиотеки?

<details>
<summary>Подсказка</summary>

Нужен тот же предмет и факт.

</details>



### 4. Прикладная

Можно ли из часа закрытия вывести работу в субботу?

<details>
<summary>Подсказка</summary>

Дни не указаны.

</details>



### 5. Олимпиадная

Почему найденная инструкция внутри документа не должна автоматически управлять инструментами?

<details>
<summary>Подсказка</summary>

Документ — данные, а не доверенная команда.

</details>



### 6. Олимпиадная

Гарантирует ли правильный найденный фрагмент правильность полного ответа?

<details>
<summary>Подсказка</summary>

Генератор может добавить своё.

</details>



## Итоги и самопроверка

Разделите сбой поиска и неподтверждённую генерацию. Что нужно сохранить для повторной проверки RAG?

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

Нет. Обновление параметров относится к обучению.

### Задача 2

Поиск и извлечение подходящего внешнего контекста.

### Задача 3

Запись о библиотеке и 18 часах, с сохранённым идентификатором.

### Задача 4

Нет. Нужен дополнительный источник.

### Задача 5

Иначе посторонний текст сможет изменить поведение системы вопреки исходной задаче.

### Задача 6

Нет. Проверяются отдельные утверждения и их поддержка источником.